In [1]:
import pandas as pd
from tqdm import tqdm
import os
from datetime import datetime

In [2]:
df = pd.read_csv('./Embedding_Integrated_128.csv', encoding='utf-8-sig')
df.head()

,Unnamed: 0,출처,키워드,제목,내용,작성일,링크,작성연도,작성월,token,bigram,doc2vec,kure_embedding,gemi_embedding
0,0,네이버카페_레몬테라스,육아일기,돌아기 전집 뭐가 좋나요,이제 돌 지나서 돌 전집 사주려고 하는데요 프뢰벨 영아다중이랑 말하기 이렇게만 있거...,2024-12-31,https://cafe.naver.com/f-e/cafes/10298136/arti...,2024,12,"['돌', '지나다', '돌', '전', '집', '사다', '프뢰벨', '영아',...","['돌_지나다', '지나다_돌', '돌_전', '전_집', '집_사다', '사다_프...",[0.01753912 5.2253723 6.9296565 3.2031174 6...,[ 8.4111039e-03 -2.8329432e-02 1.0804157e+01 ...,[10.758352 4.998933 9.212615 8.782161 ...
1,1,블로그_네이버,임신,하지불안증후군 증상 치료 원인 완치가능성,하지불안증후군 다리에서 불쾌한 감각을 느끼고 이를 해소하기 위하여 움직이고 싶은 강...,2024-12-31,https://blog.naver.com/rkfkadk04/223710185729,2024,12,"['하지불안 증후군', '다리', '감각', '느끼다', '해소', '위하다', '...","['하지불안 증후군_다리', '다리_감각', '감각_느끼다', '느끼다_해소', '...",[0.01918965 1.5984291 4.866239 2.3373353 5...,[ 0.02329755 0.33748412 5.701536 10.034008...,[ 8.288874 -1.9306369 4.573633 4.8890815 ...
2,2,네이버카페_맘이베베,육아질문방,주차 튼살크림 추천해주세요,튼살크림이 종류가 엄청 많더라구요 그래서 알아보고 있는데 종류가 너무 다양해서 고민...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,2024,12,"['트다', '살', '크림', '종류', '많다', '알아보다', '종류', '다...","['트다_살', '살_크림', '크림_종류', '종류_많다', '많다_알아보다', ...",[0.0205812 3.1714258 4.1159425 2.909176 6.192...,[4.2346669e-03 1.8436287e-02 9.1599712e+00 8.8...,[ 8.7344265 -0.7257305 9.813364 3.1906106 ...
3,3,블로그_네이버,육아,셀프 산후조리 시작 완모 산후도우미 불러야할까,셀프 산후조리 시작 완모 산후도우미 불러야할까 오늘 입원 마지막 날 우리는 셋이 되...,2024-12-31,https://blog.naver.com/today_h/223711009087,2024,12,"['셀프', '산후', '조리', '시작', '완', '산후', '도우미', '부르...","['셀프_산후', '산후_조리', '조리_시작', '시작_완', '완_산후', '산...",[0.02245066 2.3978002 6.758605 3.3574617 5...,[0.02605256 0.21050377 7.150107 9.429928 1...,[8.98485 0.24784486 9.430875 6.4761147 8...
4,4,네이버카페_맘이베베,육아질문방,입덧약 처방 받아야 할지 좀 참아볼지 고민입니다,둘째는 입덧이 다르네요 음식을 먹고난 직후가 제일 심하고 냄새에 엄청 예민해지고 체...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,2024,12,"['입덧', '다르다', '음식', '먹다', '직후', '심하다', '냄새', '...","['입덧_다르다', '다르다_음식', '음식_먹다', '먹다_직후', '직후_심하다...",[0.01910518 1.5151522 4.874107 1.3852907 5...,[2.0861698e-03 7.1943946e-02 8.5836906e+00 1.0...,[ 8.024213 -2.1786668 10.120676 4.660753 ...


In [3]:
df.columns

Index(['Unnamed: 0', '출처', '키워드', '제목', '내용', '작성일', '링크', '작성연도', '작성월',
       'token', 'bigram', 'doc2vec', 'kure_embedding', 'gemi_embedding'],
      dtype='object')

In [4]:
df_dov2vec = df[['출처', '키워드', '제목', '내용', '작성일', '링크', 'token', 'doc2vec']]
df_dov2vec.head()

,출처,키워드,제목,내용,작성일,링크,token,doc2vec
0,네이버카페_레몬테라스,육아일기,돌아기 전집 뭐가 좋나요,이제 돌 지나서 돌 전집 사주려고 하는데요 프뢰벨 영아다중이랑 말하기 이렇게만 있거...,2024-12-31,https://cafe.naver.com/f-e/cafes/10298136/arti...,"['돌', '지나다', '돌', '전', '집', '사다', '프뢰벨', '영아',...",[0.01753912 5.2253723 6.9296565 3.2031174 6...
1,블로그_네이버,임신,하지불안증후군 증상 치료 원인 완치가능성,하지불안증후군 다리에서 불쾌한 감각을 느끼고 이를 해소하기 위하여 움직이고 싶은 강...,2024-12-31,https://blog.naver.com/rkfkadk04/223710185729,"['하지불안 증후군', '다리', '감각', '느끼다', '해소', '위하다', '...",[0.01918965 1.5984291 4.866239 2.3373353 5...
2,네이버카페_맘이베베,육아질문방,주차 튼살크림 추천해주세요,튼살크림이 종류가 엄청 많더라구요 그래서 알아보고 있는데 종류가 너무 다양해서 고민...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,"['트다', '살', '크림', '종류', '많다', '알아보다', '종류', '다...",[0.0205812 3.1714258 4.1159425 2.909176 6.192...
3,블로그_네이버,육아,셀프 산후조리 시작 완모 산후도우미 불러야할까,셀프 산후조리 시작 완모 산후도우미 불러야할까 오늘 입원 마지막 날 우리는 셋이 되...,2024-12-31,https://blog.naver.com/today_h/223711009087,"['셀프', '산후', '조리', '시작', '완', '산후', '도우미', '부르...",[0.02245066 2.3978002 6.758605 3.3574617 5...
4,네이버카페_맘이베베,육아질문방,입덧약 처방 받아야 할지 좀 참아볼지 고민입니다,둘째는 입덧이 다르네요 음식을 먹고난 직후가 제일 심하고 냄새에 엄청 예민해지고 체...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,"['입덧', '다르다', '음식', '먹다', '직후', '심하다', '냄새', '...",[0.01910518 1.5151522 4.874107 1.3852907 5...


In [5]:
df_kure_emb = df[['출처', '키워드', '제목', '내용', '작성일', '링크', 'token','kure_embedding']]
df_kure_emb.head()

,출처,키워드,제목,내용,작성일,링크,token,kure_embedding
0,네이버카페_레몬테라스,육아일기,돌아기 전집 뭐가 좋나요,이제 돌 지나서 돌 전집 사주려고 하는데요 프뢰벨 영아다중이랑 말하기 이렇게만 있거...,2024-12-31,https://cafe.naver.com/f-e/cafes/10298136/arti...,"['돌', '지나다', '돌', '전', '집', '사다', '프뢰벨', '영아',...",[ 8.4111039e-03 -2.8329432e-02 1.0804157e+01 ...
1,블로그_네이버,임신,하지불안증후군 증상 치료 원인 완치가능성,하지불안증후군 다리에서 불쾌한 감각을 느끼고 이를 해소하기 위하여 움직이고 싶은 강...,2024-12-31,https://blog.naver.com/rkfkadk04/223710185729,"['하지불안 증후군', '다리', '감각', '느끼다', '해소', '위하다', '...",[ 0.02329755 0.33748412 5.701536 10.034008...
2,네이버카페_맘이베베,육아질문방,주차 튼살크림 추천해주세요,튼살크림이 종류가 엄청 많더라구요 그래서 알아보고 있는데 종류가 너무 다양해서 고민...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,"['트다', '살', '크림', '종류', '많다', '알아보다', '종류', '다...",[4.2346669e-03 1.8436287e-02 9.1599712e+00 8.8...
3,블로그_네이버,육아,셀프 산후조리 시작 완모 산후도우미 불러야할까,셀프 산후조리 시작 완모 산후도우미 불러야할까 오늘 입원 마지막 날 우리는 셋이 되...,2024-12-31,https://blog.naver.com/today_h/223711009087,"['셀프', '산후', '조리', '시작', '완', '산후', '도우미', '부르...",[0.02605256 0.21050377 7.150107 9.429928 1...
4,네이버카페_맘이베베,육아질문방,입덧약 처방 받아야 할지 좀 참아볼지 고민입니다,둘째는 입덧이 다르네요 음식을 먹고난 직후가 제일 심하고 냄새에 엄청 예민해지고 체...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,"['입덧', '다르다', '음식', '먹다', '직후', '심하다', '냄새', '...",[2.0861698e-03 7.1943946e-02 8.5836906e+00 1.0...


In [6]:
df_gem_emb = df[['출처', '키워드', '제목', '내용', '작성일', '링크', 'token','gemi_embedding']]
df_gem_emb.head()

,출처,키워드,제목,내용,작성일,링크,token,gemi_embedding
0,네이버카페_레몬테라스,육아일기,돌아기 전집 뭐가 좋나요,이제 돌 지나서 돌 전집 사주려고 하는데요 프뢰벨 영아다중이랑 말하기 이렇게만 있거...,2024-12-31,https://cafe.naver.com/f-e/cafes/10298136/arti...,"['돌', '지나다', '돌', '전', '집', '사다', '프뢰벨', '영아',...",[10.758352 4.998933 9.212615 8.782161 ...
1,블로그_네이버,임신,하지불안증후군 증상 치료 원인 완치가능성,하지불안증후군 다리에서 불쾌한 감각을 느끼고 이를 해소하기 위하여 움직이고 싶은 강...,2024-12-31,https://blog.naver.com/rkfkadk04/223710185729,"['하지불안 증후군', '다리', '감각', '느끼다', '해소', '위하다', '...",[ 8.288874 -1.9306369 4.573633 4.8890815 ...
2,네이버카페_맘이베베,육아질문방,주차 튼살크림 추천해주세요,튼살크림이 종류가 엄청 많더라구요 그래서 알아보고 있는데 종류가 너무 다양해서 고민...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,"['트다', '살', '크림', '종류', '많다', '알아보다', '종류', '다...",[ 8.7344265 -0.7257305 9.813364 3.1906106 ...
3,블로그_네이버,육아,셀프 산후조리 시작 완모 산후도우미 불러야할까,셀프 산후조리 시작 완모 산후도우미 불러야할까 오늘 입원 마지막 날 우리는 셋이 되...,2024-12-31,https://blog.naver.com/today_h/223711009087,"['셀프', '산후', '조리', '시작', '완', '산후', '도우미', '부르...",[8.98485 0.24784486 9.430875 6.4761147 8...
4,네이버카페_맘이베베,육아질문방,입덧약 처방 받아야 할지 좀 참아볼지 고민입니다,둘째는 입덧이 다르네요 음식을 먹고난 직후가 제일 심하고 냄새에 엄청 예민해지고 체...,2024-12-31,https://cafe.naver.com/f-e/cafes/29434212/arti...,"['입덧', '다르다', '음식', '먹다', '직후', '심하다', '냄새', '...",[ 8.024213 -2.1786668 10.120676 4.660753 ...


## csv 저장

In [7]:
print("df_dov2vec 총 데이터 개수 : ", len(df_dov2vec)) # 256
print("df_kure_emb 총 데이터 개수 : ", len(df_kure_emb)) # 1024
print("df_gem_emb 총 데이터 개수 : ", len(df_gem_emb)) # 256

df_dov2vec.to_csv("./Embedding_d2v_128.csv", index=False, encoding='utf-8-sig')
df_kure_emb.to_csv("./Embedding_kure_128.csv", index=False, encoding='utf-8-sig')
df_gem_emb.to_csv("./Embedding_gem_128.csv", index=False, encoding='utf-8-sig')

df_dov2vec 총 데이터 개수 :  58222
df_kure_emb 총 데이터 개수 :  58222
df_gem_emb 총 데이터 개수 :  58222
